In [ ]:
import os
import cv2
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION ET CHEMINS (DOMAINE B)
# ==========================================
# Chemins exacts fournis
PATH_IMG_STEV = r"C:\Users\Yassin\Desktop\eye tracking\data\DomaineB\Stev22\Stev2"
PATH_CSV_STEV = r"C:\Users\Yassin\Desktop\eye tracking\data\DomaineB\Stev22\SC2_.csv"

PATH_IMG_YASS = r"C:\Users\Yassin\Desktop\eye tracking\data\DomaineB\yass\yassine"
PATH_CSV_YASS = r"C:\Users\Yassin\Desktop\eye tracking\data\DomaineB\yass\Excel_yassine.csv"

# Checkpoint du Domaine A
CHECKPOINT_A = 'gaze_model.pth'

# Paramètres
IMG_SIZE = (224, 224)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. CHARGEMENT ET FUSION DES DONNÉES
# ==========================================
print("--- 1. Chargement des données Domaine B ---")

# Chargement Stevine
try:
    df_stev = pd.read_csv(PATH_CSV_STEV)
    df_stev['full_path'] = df_stev['frame_id'].apply(lambda x: os.path.join(PATH_IMG_STEV, f"frame_{int(x):04d}.png"))
    print(f"Stevine : {len(df_stev)} lignes.")
except Exception as e:
    print(f"Erreur Stevine : {e}")
    df_stev = pd.DataFrame()

# Chargement Yassine
try:
    # NOTE : Si ton fichier est un vrai .xlsx, change read_csv par read_excel
    df_yass = pd.read_csv(PATH_CSV_YASS) 
    df_yass['full_path'] = df_yass['frame_id'].apply(lambda x: os.path.join(PATH_IMG_YASS, f"frame_{int(x):04d}.png"))
    print(f"Yassine : {len(df_yass)} lignes.")
except Exception as e:
    print(f"Erreur Yassine : {e}")
    df_yass = pd.DataFrame()

# Fusion
df_full = pd.concat([df_stev, df_yass], ignore_index=True)

# Vérification existence fichiers
valid_rows = []
print("Vérification des fichiers images (cela peut prendre quelques secondes)...")
for idx, row in df_full.iterrows():
    if os.path.exists(row['full_path']):
        valid_rows.append(row)

df_final = pd.DataFrame(valid_rows).reset_index(drop=True)
TOTAL_IMGS = len(df_final)
print(f"-> Total images valides trouvées : {TOTAL_IMGS}")

if TOTAL_IMGS == 0:
    print("ERREUR : Aucune image trouvée. Vérifie tes chemins !")
    exit()

# ==========================================
# 3. DIVISION AUTOMATIQUE (SPLIT)
# ==========================================
# On garde 20% pour le Test, 80% pour le Pool
df_pool, df_test = train_test_split(df_final, test_size=0.20, random_state=42, shuffle=True)

print("\n--- 2. Résultat de la Division (Split) ---")
print(f"TEST SET (Fixe)      : {len(df_test)} images (utilisées pour noter le modèle)")
print(f"POOL SET (Entraînable): {len(df_pool)} images (utilisées pour apprendre)")
print("------------------------------------------")

# ==========================================
# 4. DATASET ET MODÈLE
# ==========================================
class EyeTrackingDatasetB(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['full_path'])
        img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), IMG_SIZE)
        # Normalisation comme Domaine A
        label = torch.tensor([row['x']/1272.0, abs(row['y'])/712.0], dtype=torch.float32)
        img_t = transforms.Normalize([0.5]*3, [0.5]*3)(transforms.ToTensor()(img))
        return img_t, label

class GazeNet(nn.Module):
    def __init__(self):
        super(GazeNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4, 4),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(4, 4)
        )
        self.regressor = nn.Sequential(
            nn.Linear(32 * 14 * 14, 256), nn.ReLU(),
            nn.Linear(256, 2), nn.Sigmoid()
        )
    def forward(self, x): return self.regressor(self.features(x).view(x.size(0), -1))

# ==========================================
# 5. SIMULATION ACTIVE LEARNING (RANDOM VS BASELINE)
# ==========================================
# Paliers d'ajout de données
steps = [0, 250, 500, 1000, 2000, 4000] # Adapté à ta taille de 6000
mae_results = []

test_loader = DataLoader(EyeTrackingDatasetB(df_test), batch_size=32, shuffle=False)
criterion = nn.MSELoss()

print("\n--- 3. Lancement de la Simulation ---")

for n in steps:
    # A. Reset du Modèle (On repart toujours du modèle Domaine A)
    model = GazeNet().to(DEVICE)
    if os.path.exists(CHECKPOINT_A):
        model.load_state_dict(torch.load(CHECKPOINT_A))
    else:
        print(f"Erreur : {CHECKPOINT_A} introuvable.")
        break

    # B. Entraînement (Sauf pour n=0 qui est la Baseline)
    if n > 0:
        # On pioche n images au hasard dans le Pool
        if n > len(df_pool): n = len(df_pool)
        subset = df_pool.sample(n=n, random_state=42)
        train_loader = DataLoader(EyeTrackingDatasetB(subset), batch_size=32, shuffle=True)
        
        optimizer = optim.Adam(model.parameters(), lr=0.0005)
        model.train()
        # Petit Fine-Tuning (3 époques suffisent pour voir l'adaptation)
        for _ in range(3): 
            for imgs, lbls in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(imgs.to(DEVICE)), lbls.to(DEVICE))
                loss.backward(); optimizer.step()
    
    # C. Test sur le Test Set Fixe
    model.eval()
    total_mae = 0
    with torch.no_grad():
        for imgs, lbls in test_loader:
            out = model(imgs.to(DEVICE))
            total_mae += torch.mean(torch.abs(out - lbls.to(DEVICE))).item() * imgs.size(0)
    
    final_mae = total_mae / len(df_test)
    mae_results.append(final_mae)
    
    label_step = "BASELINE (0 img)" if n == 0 else f"Random ({n} img)"
    print(f"[{label_step}] -> MAE : {final_mae:.4f}")

# ==========================================
# 6. GRAPHIQUE FINAL
# ==========================================
baseline_val = mae_results[0]

plt.figure(figsize=(10, 6))
# Ligne rouge horizontale pour la Baseline
plt.axhline(y=baseline_val, color='r', linestyle='--', label=f'Baseline (Domaine A) = {baseline_val:.4f}')
# Courbe bleue pour l'amélioration Random
plt.plot(steps, mae_results, marker='o', linestyle='-', color='b', label='Stratégie Aléatoire')

plt.xlabel("Nombre d'images ajoutées du Domaine B")
plt.ylabel("Erreur Moyenne (MAE)")
plt.title(f"Adaptation au Domaine B (Test Set: {len(df_test)} images)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nTerminé ! Tu as ta courbe Baseline vs Random.")